<a href="https://colab.research.google.com/github/DurgaPrasad-1805/Quantum-Computing-Project/blob/main/Colab_Quantum%20Computing%20Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install qiskit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 kB 2.0 MB/s eta 0:00:00


In [ ]:
!pip install qiskit qiskit-aer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 139.4 MB/s eta 0:00:00


In [ ]:
# Simon's Algorithm implementation (Qiskit 1.x compatible)

!pip install qiskit qiskit-aer -q

import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.circuit.library import UnitaryGate
from math import log2
from collections import Counter

def make_simon_f_map(n, s):
    N = 1 << n
    s_int = int(s, 2)
    used = [False]*N
    f = [None]*N
    next_label = 0
    for x in range(N):
        if not used[x]:
            partner = x ^ s_int
            f[x] = next_label
            f[partner] = next_label
            used[x] = used[partner] = True
            next_label += 1
    output_vals = list(range(1 << (n-1)))
    f_out = [output_vals[label] for label in f]
    return f_out

def build_oracle_unitary(n, f_out):
    N = 1 << n
    dim = 1 << (2*n)
    U = np.zeros((dim, dim), dtype=complex)
    for x in range(N):
        fx = f_out[x]
        for y in range(N):
            in_index = (x << n) | y
            out_y = y ^ fx
            out_index = (x << n) | out_y
            U[out_index, in_index] = 1.0
    return UnitaryGate(U, label="U_f")

def gf2_solve(eqs, n):
    m = len(eqs)
    A = [list(map(int, list(format(eqs[i], f'0{n}b')))) for i in range(m)]
    row = 0
    pivots = []
    for col in range(n):
        pivot = None
        for r in range(row, m):
            if A[r][col] == 1:
                pivot = r
                break
        if pivot is None:
            continue
        A[row], A[pivot] = A[pivot], A[row]
        pivots.append(col)
        for r in range(m):
            if r != row and A[r][col] == 1:
                for c in range(col, n):
                    A[r][c] ^= A[row][c]
        row += 1
        if row == m:
            break
    rank = len(pivots)
    null_dim = n - rank
    if null_dim == 0:
        return 0
    pivot_set = set(pivots)
    free_cols = [c for c in range(n) if c not in pivot_set]
    free_choice = free_cols[0]
    s = [0]*n
    s[free_choice] = 1
    pivot_row_for_col = {c: i for i, c in enumerate(pivots)}
    for c in pivots[::-1]:
        row_idx = pivot_row_for_col[c]
        total = 0
        for j in range(c+1, n):
            total ^= (A[row_idx][j] & s[j])
        s[c] = total
    s_int = int("".join(map(str, s)), 2)
    return s_int

def simon_algorithm(n, s_bin, shots=1024):
    f = make_simon_f_map(n, s_bin)
    U = build_oracle_unitary(n, f)
    total_qubits = 2*n
    qc = QuantumCircuit(total_qubits, n)
    input_q = list(range(n))
    output_q = list(range(n, 2*n))
    qc.h(input_q)
    qc.append(U, list(range(total_qubits)))
    qc.h(input_q)
    qc.measure(input_q, range(n))

    simulator = AerSimulator()
    job = simulator.run(qc, shots=shots)
    result = job.result()
    counts = result.get_counts(qc)

    measurement_ints = [int(k, 2) for k in counts.keys()]
    samples = []
    for k, v in counts.items():
        samples += [int(k, 2)] * v
    unique_z = sorted(set(samples))
    eqs = [z for z in unique_z if z != 0]
    s_found = gf2_solve(eqs, n)
    return s_found, counts, qc

# ---------- Manual Input ----------
print("=== Simon's Algorithm Simulation ===")
n = int(input("Enter number of qubits (e.g. 2,3,4,...): "))
s_bin = input(f"Enter secret string s (length {n}): ")
if len(s_bin) != n or not all(c in '01' for c in s_bin):
    raise ValueError("Invalid secret string: must be binary and of length n")

# ---------- Run Algorithm ----------
s_found, counts, qc = simon_algorithm(n, s_bin)

print("\nMeasurement Counts:")
for k, v in counts.items():
    print(f"{k}: {v}")

print(f"\nOriginal secret string s: {s_bin}")
print(f"Recovered secret string s': {format(s_found, f'0{n}b')}")

print("\nQuantum Circuit:")
print(qc.draw(output='text'))

# ===================== Interactive Example Section =====================

#n = int(input("Enter the number of qubits (2, 3, or 4): "))

if n == 2:
    s_bin = "10"
    mapping = {
        "00": "01",
        "01": "11",
        "10": "01",
        "11": "11"
    }

    #print(f"\n=== Example: n = {n}, s = {s_bin} ===")
    print("Now define a 2-to-1 function f(x) such that f(x) = f(x ⊕ s):\n")
    print("x\tf(x)")
    for x, fx in mapping.items():
        print(f"{x}\t{fx}")
    print("\nCheck:\n• f(00) = f(10) = 01\n• f(01) = f(11) = 11")

elif n == 3:
    s_bin = "101"
    mapping = {
        "000": "001",
        "001": "111",
        "010": "010",
        "011": "100",
        "100": "001",
        "101": "111",
        "110": "010",
        "111": "100"
    }

    #print(f"\n=== Example: n = {n}, s = {s_bin} ===")
    print("Now define a 2-to-1 function f(x) such that f(x) = f(x ⊕ s):\n")
    print("x\tf(x)")
    for x, fx in mapping.items():
        print(f"{x}\t{fx}")
    print("\nCheck:\n• f(000)=f(100)=001\n• f(001)=f(101)=111\n• f(010)=f(110)=010\n• f(011)=f(111)=100")

elif n == 4:
    s_bin = "1100"
    mapping = {
        "0000": "0001",
        "0001": "1010",
        "0010": "0100",
        "0011": "1111",
        "0100": "0001",
        "0101": "1010",
        "0110": "0100",
        "0111": "1111",
        "1000": "0011",
        "1001": "1100",
        "1010": "0110",
        "1011": "1001",
        "1100": "0011",
        "1101": "1100",
        "1110": "0110",
        "1111": "1001"
    }

    #print(f"\n=== Example: n = {n}, s = {s_bin} ===")
    print("Now define a 2-to-1 function f(x) such that f(x) = f(x ⊕ s):\n")
    print("x\tf(x)")
    for x, fx in mapping.items():
        print(f"{x}\t{fx}")
    print("\nCheck:\n• f(0000)=f(1100)=0001\n• f(0001)=f(1101)=1010\n• f(0010)=f(1110)=0100\n• f(0011)=f(1111)=1111")

else:
    print("\nInvalid input! Please enter n = 2, 3, or 4.")
    quit()



=== Simon's Algorithm Simulation ===
Enter number of qubits (e.g. 2,3,4,...): 2
Enter secret string s (length 2): 10

Measurement Counts:
00: 1024

Original secret string s: 10
Recovered secret string s': 10

Quantum Circuit:
     ┌───┐┌──────┐┌───┐┌─┐   
q_0: ┤ H ├┤0     ├┤ H ├┤M├───
     ├───┤│      │├───┤└╥┘┌─┐
q_1: ┤ H ├┤1     ├┤ H ├─╫─┤M├
     └───┘│  U_f │└───┘ ║ └╥┘
q_2: ─────┤2     ├──────╫──╫─
          │      │      ║  ║ 
q_3: ─────┤3     ├──────╫──╫─
          └──────┘      ║  ║ 
c: 2/═══════════════════╩══╩═
                        0  1 
Now define a 2-to-1 function f(x) such that f(x) = f(x ⊕ s):

x	f(x)
00	01
01	11
10	01
11	11

Check:
• f(00) = f(10) = 01
• f(01) = f(11) = 11
